# 💬 Multi-turn Chat — remembering the conversation

Notebook **03** in the pi.dev series. In [`01`](./01-hello-world.ipynb) we sent one
message. Real chat needs **memory**: the model itself is stateless, so *we* keep the
history and replay it every turn.

The key idea: a conversation is just a growing **`Message[]`** on `Context.messages`.
Each turn you (1) append the user's `UserMessage`, (2) call the model, then (3) push the
returned `AssistantMessage` **object** straight back onto `messages` — and repeat. Because
the whole transcript is re-sent each time, the model "remembers" earlier turns (and the
token cost grows with the history).

> **Kernel:** **Deno** (pi.dev is ESM-only TypeScript). See `../README.md`.

## 1. Load env & register Azure OpenAI

Same two-step bootstrap every notebook uses: `loadEnvUp()` pulls `AZURE_PI_TEST_*` from a
`.env` outside the repo, then `registerAzure()` wires up the Azure OpenAI provider and hands
back a ready-to-use `model`.

In [ ]:
import { loadEnvUp } from "playground/env";
const env = await loadEnvUp();

In [ ]:
import { registerAzure } from "playground/azure";
const { models, model, modelId } = registerAzure();

## 2. A conversation is just a growing message list

We build one **mutable** `context` and a small `ask()` helper that runs a single turn:

1. push the user's message,
2. `completeSimple(model, context)` — the model sees the **entire** history,
3. push the assistant's reply object back on,
4. tally running tokens & cost.

`textOf()` pulls the plain text out of a message, whose `content` is either a `string`
(user) or an array of content blocks (assistant).

In [ ]:
import { type Context, type Message } from "@earendil-works/pi-ai";

// One mutable conversation. `messages` grows as we talk.
const context: Context = {
  systemPrompt: "You are a concise, friendly assistant. Keep answers to 1-2 short sentences.",
  messages: [],
};

// Running totals so we can watch cost accumulate turn over turn.
let totalTokens = 0;
let totalCost = 0;

// Extract the plain text from any message (content is a string or content blocks).
function textOf(msg: Message): string {
  const c = msg.content;
  if (typeof c === "string") return c;
  return c
    .filter((b): b is { type: "text"; text: string } => b.type === "text")
    .map((b) => b.text)
    .join("");
}

// Run one turn: append user message, complete, append reply, print + tally.
async function ask(text: string): Promise<void> {
  context.messages.push({ role: "user", content: text, timestamp: Date.now() });

  const reply = await models.completeSimple(model, context);
  context.messages.push(reply); // <-- this is what gives the chat its memory

  totalTokens += reply.usage.totalTokens;
  totalCost += reply.usage.cost.total;

  console.log(`🧑 ${text}`);
  console.log(`🤖 ${textOf(reply)}`);
  console.log(
    `   [stop: ${reply.stopReason}]  turn ${reply.usage.input} in / ${reply.usage.output} out` +
    `  |  running ${totalTokens} tok, $${totalCost.toFixed(6)}\n`,
  );
}

console.log(`Ready to chat with ${modelId}.`);

## 3. Three turns that build on each other

Turn 2 asks about something only stated in turn 1, and turn 3 depends on both. Nothing is
passed between calls except the shared `context.messages` list — that replayed history is
the model's entire memory.

In [ ]:
await ask("My name is Gian and I love sailing.");
await ask("What's my name?");
await ask("Based on what I like, suggest one activity for this weekend.");

## 4. Inspect the transcript & cumulative cost

After three turns the list holds **6 messages** (user + assistant, ×3). The cumulative cost
is higher than 3× a single turn would be, because every turn re-sends the *whole* transcript
as input — the price of statelessness.

In [ ]:
console.log(`transcript has ${context.messages.length} messages:\n`);
for (const m of context.messages) {
  const who = m.role === "user" ? "🧑 user" : m.role === "assistant" ? "🤖 asst" : "🔧 tool";
  const line = textOf(m).replace(/\s+/g, " ").slice(0, 80);
  console.log(`${who}: ${line}`);
}
console.log(`\ncumulative: ${totalTokens} tokens, $${totalCost.toFixed(6)}`);
console.log("Cost grows each turn because the ENTIRE history is re-sent as input.");

## ✅ What just happened

1. We kept one mutable `context` with a growing `messages` array.
2. Each `ask()` appended the user turn, called `completeSimple`, and pushed the returned
   `AssistantMessage` **object** back on — so the next call replays the full history.
3. The model answered "What's my name?" and tailored the weekend suggestion purely from that
   replayed context; it holds no state of its own.
4. Running token/cost totals showed the cost of re-sending history every turn.

### Takeaways

- **You** own the memory: persist `context.messages` (e.g. to disk) to resume a chat later.
- History grows unbounded — eventually you truncate or summarize old turns to stay under the
  model's `contextWindow` and control cost. (pi's agent core automates this; see later notebooks.)

Next: **04 — Tool calling**, where the assistant asks *us* to run functions.